# CISC7204 Assignment 01 - Module 05 - Lab02 Model Evaluation and Refinement - Laptop Pricing


In [1]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod1.csv"
df = pd.read_csv(DATA_URL)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
pd.set_option("display.max_columns", None)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully.
Shape: (238, 12)


  Manufacturer  Category     Screen  GPU  OS  CPU_core  Screen_Size_cm  \
0         Acer         4  IPS Panel    2   1         5          35.560   
1         Dell         3    Full HD    1   1         3          39.624   
2         Dell         3    Full HD    1   1         7          39.624   
3         Dell         4  IPS Panel    2   1         5          33.782   
4           HP         4    Full HD    2   1         7          39.624   

   CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_kg  Price  
0            1.6       8             256       1.60    978  
1            2.0       4             256       2.20    634  
2            2.7       8             256       2.20    946  
3            1.6       8             128       1.22   1244  
4            1.8       8             256       1.91    837  

## Prepare A Clean Laptop Dataset

In [2]:
laptop_df = df.copy()

laptop_df["Screen_Size_cm"] = pd.to_numeric(laptop_df["Screen_Size_cm"], errors="coerce")
laptop_df["Weight_kg"] = pd.to_numeric(laptop_df["Weight_kg"], errors="coerce")

laptop_df["Weight_kg"].fillna(laptop_df["Weight_kg"].mean(), inplace=True)
laptop_df["Screen_Size_cm"].fillna(laptop_df["Screen_Size_cm"].mode()[0], inplace=True)

laptop_df["Screen_Size_inch"] = laptop_df["Screen_Size_cm"] / 2.54
laptop_df["Weight_pounds"] = laptop_df["Weight_kg"] * 2.205
laptop_df["CPU_frequency"] = laptop_df["CPU_frequency"] / laptop_df["CPU_frequency"].max()

bins = np.linspace(laptop_df["Price"].min(), laptop_df["Price"].max(), 4)
labels = ["Low", "Medium", "High"]
laptop_df["Price-binned"] = pd.cut(laptop_df["Price"], bins=bins, labels=labels, include_lowest=True)

screen_dummies = pd.get_dummies(laptop_df["Screen"], prefix="Screen")
laptop_df = pd.concat([laptop_df.drop(columns=["Screen"]), screen_dummies], axis=1)

print("Missing values after cleaning:")
print(laptop_df.isna().sum())
laptop_df.head()

Missing values after cleaning:
Manufacturer        0
Category            0
GPU                 0
OS                  0
CPU_core            0
Screen_Size_cm      0
CPU_frequency       0
RAM_GB              0
Storage_GB_SSD      0
Weight_kg           0
Price               0
Screen_Size_inch    0
Weight_pounds       0
Price-binned        0
Screen_Full HD      0
Screen_IPS Panel    0
dtype: int64


  Manufacturer  Category  GPU  OS  CPU_core  Screen_Size_cm  CPU_frequency  \
0         Acer         4    2   1         5          35.560       0.551724   
1         Dell         3    1   1         3          39.624       0.689655   
2         Dell         3    1   1         7          39.624       0.931034   
3         Dell         4    2   1         5          33.782       0.551724   
4           HP         4    2   1         7          39.624       0.620690   

   RAM_GB  Storage_GB_SSD  Weight_kg  Price  Screen_Size_inch  Weight_pounds  \
0       8             256       1.60    978              14.0        3.52800   
1       4             256       2.20    634              15.6        4.85100   
2       8             256       2.20    946              15.6        4.85100   
3       8             128       1.22   1244              13.3        2.69010   
4       8             256       1.91    837              15.6        4.21155   

  Price-binned  Screen_Full HD  Screen_IPS Panel  

## Task 1: Train/Test Split

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

features = ["CPU_frequency", "RAM_GB", "Storage_GB_SSD", "CPU_core", "OS", "GPU", "Category"]
X = laptop_df[features]
y = laptop_df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (190, 7)
Testing shape: (48, 7)


## Task 2: Baseline Multiple Linear Regression

In [4]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)

train_pred = baseline.predict(X_train)
test_pred = baseline.predict(X_test)

print("Train MSE:", round(mean_squared_error(y_train, train_pred), 3))
print("Train R2:", round(r2_score(y_train, train_pred), 3))
print("Test MSE:", round(mean_squared_error(y_test, test_pred), 3))
print("Test R2:", round(r2_score(y_test, test_pred), 3))

Train MSE: 165297.185
Train R2: 0.521
Test MSE: 168575.62
Test R2: 0.269


## Task 3: Cross Validation

In [5]:
cv_scores = cross_val_score(baseline, X, y, cv=4, scoring="r2")
print("4-fold CV R2 scores:", cv_scores)
print("Mean CV R2:", round(cv_scores.mean(), 3))
print("Standard deviation:", round(cv_scores.std(), 3))

4-fold CV R2 scores: [-0.36871432  0.58494587  0.43347433  0.35218353]
Mean CV R2: 0.25
Standard deviation: 0.367


## Task 4: Ridge Regression

In [6]:
ridge = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

print("Ridge Test MSE:", round(mean_squared_error(y_test, ridge_pred), 3))
print("Ridge Test R2:", round(r2_score(y_test, ridge_pred), 3))

Ridge Test MSE: 167964.975
Ridge Test R2: 0.271


## Task 5: Grid Search With Ridge Regression

In [7]:
ridge_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("ridge", Ridge())
])

param_grid = {"ridge__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]}
grid = GridSearchCV(ridge_pipeline, param_grid=param_grid, cv=4, scoring="r2")
grid.fit(X_train, y_train)

best_pred = grid.predict(X_test)

print("Best alpha:", grid.best_params_["ridge__alpha"])
print("Best CV R2:", round(grid.best_score_, 3))
print("Test MSE:", round(mean_squared_error(y_test, best_pred), 3))
print("Test R2:", round(r2_score(y_test, best_pred), 3))

Best alpha: 10
Best CV R2: 0.471
Test MSE: 233541.993
Test R2: -0.013


## Task 6: Final Model Comparison

In [8]:
comparison = pd.DataFrame([
    {
        "Model": "Baseline Multiple Linear Regression",
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Test R2": r2_score(y_test, test_pred),
    },
    {
        "Model": "Ridge Regression",
        "Test MSE": mean_squared_error(y_test, ridge_pred),
        "Test R2": r2_score(y_test, ridge_pred),
    },
    {
        "Model": "Polynomial Ridge with Grid Search",
        "Test MSE": mean_squared_error(y_test, best_pred),
        "Test R2": r2_score(y_test, best_pred),
    },
])
comparison

                                 Model       Test MSE   Test R2
0  Baseline Multiple Linear Regression  168575.620438  0.268538
1                     Ridge Regression  167964.975435  0.271188
2    Polynomial Ridge with Grid Search  233541.993000 -0.013355